In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce.gold

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Define variáveis de catálogo e schemas de origem (silver) e destino (gold)
catalog = "ecommerce"
schema = "silver"
newSchema = "gold"

# Carrega todas as tabelas da camada silver 
consumidores_silver_df = spark.table(f"{catalog}.{schema}.dim_consumidores")
pedidos_silver_df = spark.table(f"{catalog}.{schema}.fat_pedidos")
itens_pedidos_silver_df = spark.table(f"{catalog}.{schema}.fat_itens_pedidos")
pagamentos_pedidos_silver_df = spark.table(f"{catalog}.{schema}.fat_pagamentos_pedidos")
avaliacoes_pedidos_silver_df = spark.table(f"{catalog}.{schema}.fat_avaliacoes_pedidos")
produtos_silver_df = spark.table(f"{catalog}.{schema}.dim_produtos")
vendedores_silver_df = spark.table(f"{catalog}.{schema}.dim_vendedores")
cotacao_dolar_silver_df = spark.table(f"{catalog}.{schema}.dim_cotacao_dolar")
categoria_produtos_traducao_silver_df = spark.table(f"{catalog}.{schema}.dim_categoria_produtos_traducao")

print("Tabelas Silver carregadas!")

Tabelas Silver carregadas!


In [0]:
print("Consumidores silver:")
consumidores_silver_df.printSchema()
print("Pedidos silver:")
pedidos_silver_df.printSchema()
print("Itens pedidos silver:") 
itens_pedidos_silver_df.printSchema()
print("Pagamentos pedidos silver:")
pagamentos_pedidos_silver_df.printSchema()
print("Avaliações pedidos silver:")
avaliacoes_pedidos_silver_df.printSchema()
print("Produtos silver:")
produtos_silver_df.printSchema()
print("Vendedores silver:")
vendedores_silver_df.printSchema()
print("Cotação dólar silver:")
cotacao_dolar_silver_df.printSchema()
print("Categoria produtos tradução silver:")
categoria_produtos_traducao_silver_df.printSchema()

Consumidores silver:
root
 |-- idade: integer (nullable = true)
 |-- data_nascimento: date (nullable = true)
 |-- cidade: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- id_consumidor: string (nullable = true)
 |-- nome_consumidor: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- id_unico: string (nullable = true)
 |-- prefixo_cep: integer (nullable = true)

Pedidos silver:
root
 |-- id_pedido: string (nullable = true)
 |-- id_consumidor: string (nullable = true)
 |-- status: string (nullable = true)
 |-- data_pedido: timestamp (nullable = true)
 |-- data_aprovacao: timestamp (nullable = true)
 |-- data_entrega_transportadora: timestamp (nullable = true)
 |-- data_entrega_consumidor: timestamp (nullable = true)
 |-- data_entrega_estimada: timestamp (nullable = true)
 |-- tempo_entrega_dias: integer (nullable = true)
 |-- tempo_entrega_estimado_dias: integer (nullable = true)
 |-- diferenca_entrega_dias: integer (nullable = true)
 |-- entrega_no_pr

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{newSchema}.fat_vendas_comercial")
# Remove a tabela fat_vendas_comercial se já existir, para recriá-la do zero.

pedidos_enriquecidos_df = (
    pedidos_silver_df
    .withColumn("data_pedido_date", F.to_date("data_pedido"))
    # Cria uma coluna apenas com a data (sem horário) do pedido para facilitar joins.
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
    # Extrai ano e mês da data do pedido para usar na agregação comercial.
)

df_join = (
    pedidos_enriquecidos_df
    .join(itens_pedidos_silver_df, on="id_pedido", how="left")
    # Junta pedidos com seus itens para obter valores de preço/frete por item.
    .join(
        produtos_silver_df
        .select("id_produto", "categoria_produto"), 
        on="id_produto", 
        how="left"
    )
    # Adiciona a categoria de produto aos itens vendidos.
    .join(
        cotacao_dolar_silver_df
        .select("data_cotacao", "cotacao_compra"), 
        F.col("data_pedido_date") == F.col("data_cotacao"),
        how="left"
    )
    # Junta a cotação do dólar da data do pedido para permitir conversão BRL → USD.
)

vendas_comercial_gold_df = (
    df_join
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    # Agrega por ano, mês e categoria de produto.
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        # Número de pedidos distintos no período/categoria.
        F.count("id_item").alias("qtd_itens_vendidos"),
        # Quantidade total de itens vendidos.
        F.sum("preco_BRL").alias("receita_total_brl_raw"),
        # Receita bruta em BRL (antes de arredondar).
        F.sum(F.col("preco_BRL") / F.col("cotacao_compra")).alias("receita_total_usd_raw")
        # Receita bruta em USD usando a cotação do dia do pedido.
    )
    .withColumn(
        "receita_total_brl", 
        F.round(F.col("receita_total_brl_raw"), 2)
    )
    # Cria coluna de receita BRL arredondada a 2 casas decimais.
    .withColumn(
        "receita_total_usd", 
        F.round(F.col("receita_total_usd_raw"), 2)
    )
    # Cria coluna de receita USD arredondada.
    .withColumn(
        "ticket_medio_brl", 
        F.round(F.col("receita_total_brl_raw") / F.col("total_pedidos"), 2)
    )
    # Calcula o ticket médio em BRL por pedido no período/categoria.
    .drop("receita_total_brl_raw", "receita_total_usd_raw")
    # Remove colunas auxiliares não arredondadas.
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
    # Ordena o resultado para facilitar consulta e visualização.
)

vendas_comercial_gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{newSchema}.fat_vendas_comercial")
# Escreve o resultado agregado como tabela Delta na camada silver (fato comercial).

spark.table(f"{catalog}.{newSchema}.fat_vendas_comercial").display()
# Exibe a tabela final de vendas comerciais para validação.

ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2016,9,null,1,0,null,null,null
2016,9,beleza_saude,1,3,134.97,40.51,134.97
2016,9,moveis_decoracao,1,2,72.89,22.48,72.89
2016,9,telefonia,1,1,59.5,18.19,59.5
2016,10,null,18,2,65.89,20.48,3.66
2016,10,alimentos,1,1,79.9,24.88,79.9
2016,10,audio,2,2,156.99,48.87,78.5
2016,10,automotivo,11,12,1833.25,568.39,166.66
2016,10,bebes,11,14,1630.16,505.5,148.2
2016,10,beleza_saude,44,48,4552.51,1414.77,103.47


In [0]:
rank_produtos_df = (
    itens_pedidos_silver_df
    .join(produtos_silver_df, on="id_produto", how="inner")
    # Junta itens de pedidos com a dimensão de produtos para ter nome e categoria.
    .groupBy("nome_produto", "categoria_produto")
    # Agrupa por produto e categoria.
    .agg(F.count("id_item").alias("quantidade_vendida"))
    # Calcula quantos itens foram vendidos de cada produto.
)

top_5_mais_vendidos_df = (
    rank_produtos_df
    .orderBy(F.col("quantidade_vendida").desc())
    # Ordena do mais vendido para o menos vendido.
    .limit(5)
    # Mantém apenas os 5 produtos mais vendidos.
)

top_5_menos_vendidos_df = (
    rank_produtos_df
    .orderBy(F.col("quantidade_vendida").asc())
    # Ordena do menos vendido para o mais vendido.
    .limit(5)
    # Mantém apenas os 5 produtos menos vendidos.
)

print("Top 5 Produtos MAIS Vendidos:")
top_5_mais_vendidos_df.display()
# Exibe os 5 produtos com maior quantidade vendida.

print("Top 5 Produtos MENOS Vendidos:")
top_5_menos_vendidos_df.display()
# Exibe os 5 produtos com menor quantidade vendida.

Top 5 Produtos MAIS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


Top 5 Produtos MENOS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Produto Genérico Preto,moveis_quarto,1
Item Básico Premium,fashion_calcados,1
Peça de Reposição Preto,telefonia_fixa,1
Item Básico Verde,industria_comercio_e_negocios,1
Peça de Reposição Plus,fashion_calcados,1


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{newSchema}.fat_avaliacoes_clientes")
# Remove a tabela fat_avaliacoes_clientes se ela já existir, para recriá-la.

df_avaliacoes_join = (
    avaliacoes_pedidos_silver_df
    .join(itens_pedidos_silver_df, on="id_pedido", how="inner")
    # Junta avaliações com os itens de pedido para relacionar avaliação e produto.
    .join(
        produtos_silver_df.select("id_produto", "categoria_produto"), 
        on="id_produto", 
        how="left"
    )
    # Adiciona a categoria de produto associada à avaliação.
    .join(
        vendedores_silver_df.select("id_vendedor", "nome_vendedor", "estado"), 
        on="id_vendedor", 
        how="left"
    )
    # Adiciona informações do vendedor (nome e estado) ligadas ao item avaliado.
)

avaliacoes_clientes_df = (
    df_avaliacoes_join
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    # Agrega por categoria de produto, vendedor e estado.
    .agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        # Quantidade total de avaliações nessa combinação.
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        # Nota média das avaliações (arredondada a 2 casas).
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        # Total de avaliações consideradas positivas (nota >= 4).
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
        # Total de avaliações consideradas negativas (nota <= 2).
    )
    .withColumn(
        "percentual_satisfacao",
        F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2)
    )
    # Calcula o percentual de satisfação como % de avaliações positivas.
    .orderBy(F.col("percentual_satisfacao").desc(), F.col("total_avaliacoes").desc())
    # Ordena primeiro por maior satisfação, depois por maior volume de avaliações.
)

avaliacoes_clientes_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{newSchema}.fat_avaliacoes_clientes")
# Salva a fato de avaliações de clientes como tabela Delta.

spark.table(f"{catalog}.{newSchema}.fat_avaliacoes_clientes").display()
# Exibe a tabela final para validação.

categoria_produto,nome_vendedor,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
moveis_decoracao,Luísa Oliveira,SP,26,4.96,26,0,100.0
moveis_decoracao,Luigi Sá,SP,25,4.68,25,0,100.0
livros_interesse_geral,Marina Sousa,RN,23,4.96,23,0,100.0
market_place,Antônio Nunes,SP,21,4.81,21,0,100.0
perfumaria,Stephany Nascimento,SP,20,4.9,20,0,100.0
beleza_saude,João Pedro da Rosa,MS,20,4.95,20,0,100.0
pet_shop,Dr. Marcos Vinicius Carvalho,SP,20,4.85,20,0,100.0
moveis_escritorio,Jade Mendonça,SP,19,4.79,19,0,100.0
perfumaria,Olivia Lopes,SP,19,4.74,19,0,100.0
automotivo,José Pedro Ramos,SP,17,4.76,17,0,100.0


In [0]:
df_produtos_avaliacoes = (
    avaliacoes_pedidos_silver_df
    .join(itens_pedidos_silver_df, on="id_pedido", how="inner")
    # Liga cada avaliação aos itens do pedido correspondente.
    .join(
        produtos_silver_df.select("id_produto", "nome_produto"),
        on="id_produto",
        how="left"
    )
    # Adiciona o nome do produto associado à avaliação.
    .groupBy("nome_produto")
    # Agrupa por produto para calcular métricas agregadas.
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
        # Nota média do produto, arredondada a 2 casas.
        F.count("id_avaliacao").alias("volume_avaliacoes")
        # Quantidade total de avaliações recebidas pelo produto.
    )
)

produto_mais_bem_avaliado = (
    df_produtos_avaliacoes
    .orderBy(F.col("nota_media").desc(), F.col("volume_avaliacoes").desc())
    # Ordena primeiro pela maior nota média, depois pelo maior volume de avaliações (desempate).
    .limit(1)
    # Mantém apenas o produto mais bem avaliado.
)

produto_menos_bem_avaliado = (
    df_produtos_avaliacoes
    .orderBy(F.col("nota_media").asc(), F.col("volume_avaliacoes").desc())
    # Ordena pela menor nota média, e em caso de empate, maior volume.
    .limit(1)
    # Mantém apenas o produto menos bem avaliado.
)

df_vendedores_avaliacoes = (
    avaliacoes_pedidos_silver_df
    .join(itens_pedidos_silver_df, on="id_pedido", how="inner")
    # Liga avaliações aos itens do pedido.
    .join(
        vendedores_silver_df.select("id_vendedor", "nome_vendedor"),
        on="id_vendedor",
        how="left"
    )
    # Adiciona o nome do vendedor associado à avaliação.
    .groupBy("nome_vendedor")
    # Agrupa por vendedor para calcular métricas de avaliação.
    .agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
        # Nota média do vendedor, arredondada a 2 casas.
        F.count("id_avaliacao").alias("volume_avaliacoes")
        # Volume total de avaliações recebidas pelo vendedor.
    )
)

vendedor_mais_bem_avaliado = (
    df_vendedores_avaliacoes
    .orderBy(F.col("nota_media").desc(), F.col("volume_avaliacoes").desc())
    # Ordena pelo maior score médio e maior volume de avaliações.
    .limit(1)
    # Mantém apenas o vendedor mais bem avaliado.
)

vendedor_menos_bem_avaliado = (
    df_vendedores_avaliacoes
    .orderBy(F.col("nota_media").asc(), F.col("volume_avaliacoes").desc())
    # Ordena pelo menor score médio e maior volume (para evitar outliers com poucas avaliações).
    .limit(1)
    # Mantém apenas o vendedor menos bem avaliado.
)

print("O Produto MAIS bem avaliado:")
produto_mais_bem_avaliado.display()
# Exibe o produto com melhor avaliação geral.

print("O Produto MENOS bem avaliado:")
produto_menos_bem_avaliado.display()
# Exibe o produto com pior avaliação geral.

print("O Vendedor MAIS bem avaliado:")
vendedor_mais_bem_avaliado.display()
# Exibe o vendedor com melhor avaliação geral.

print("O Vendedor MENOS bem avaliado:")
vendedor_menos_bem_avaliado.display()
# Exibe o vendedor com pior avaliação geral.

O Produto MAIS bem avaliado:


nome_produto,nota_media,volume_avaliacoes
Caneta Esferográfica Avançado,5.0,18


O Produto MENOS bem avaliado:


nome_produto,nota_media,volume_avaliacoes
Conjunto de Pincéis Ultra,1.0,7


O Vendedor MAIS bem avaliado:


nome_vendedor,nota_media,volume_avaliacoes
Luiz Otávio Abreu,5.0,34


O Vendedor MENOS bem avaliado:


nome_vendedor,nota_media,volume_avaliacoes
Sra. Fernanda Santos,1.0,8
